In [1]:
"""
sanear_corpus_csv.py
Atlas Social de Hortolândia — Saneamento preventivo do corpus jornalístico

Problema: campos de texto livre (nota_analista, titulo) com vírgulas internas
sem aspas quebram a contagem de colunas no CSV.

Estratégia de reconstrução:
    - As primeiras 14 colunas são seguras (sem vírgulas internas)
    - nota_analista (pos 14) pode ter vírgulas → reconstruída pelo "meio"
    - As últimas 5 colunas são seguras
    - Resultado gravado com csv.QUOTE_MINIMAL (aspas apenas onde necessário)

Uso:
    python sanear_corpus_csv.py

Configurar PASTA_CSV abaixo antes de rodar.
"""

import csv
import io
import os
import shutil
from pathlib import Path

# ── CONFIGURAR AQUI ───────────────────────────────────────────────────────────
PASTA_CSV = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\bd_externos\series_jornalisticas"
CRIAR_BACKUP = True   # cria pasta _backup/ antes de sobrescrever
# ─────────────────────────────────────────────────────────────────────────────

HEADER_ESPERADO = [
    "id_evento", "data_publicacao", "fonte", "titulo", "pagina",
    "municipio", "localidade", "tipo_camada", "dimensao_analitica",
    "tipo_relacao_variavel", "codigo_variavel", "nivel_criticidade",
    "observacao", "aproximacao_variavel", "nota_analista",
    "cod_loteamento", "nivel_confianca_loteamento", "papel_no_ciclo",
    "id_caso_pressao", "entra_ipst",
]
N_COLUNAS   = len(HEADER_ESPERADO)   # 20
PREFIX_COLS = 14   # id_evento … aproximacao_variavel
SUFFIX_COLS = 5    # cod_loteamento … entra_ipst


def reconstruir_linha(parts: list[str]) -> list[str] | None:
    """
    Reconstrói uma linha partida por vírgulas internas no nota_analista.
    Retorna lista de 20 campos ou None se não for possível reconstruir.
    """
    if len(parts) < N_COLUNAS:
        return None  # linha curta demais — problema diferente
    if len(parts) == N_COLUNAS:
        return parts  # já está certo

    prefix = parts[:PREFIX_COLS]
    suffix = parts[-SUFFIX_COLS:]
    nota   = ",".join(parts[PREFIX_COLS : len(parts) - SUFFIX_COLS])
    return prefix + [nota] + suffix


def sanear_arquivo(caminho: Path, pasta_backup: Path) -> dict:
    """
    Lê, corrige e reescreve um CSV.
    Retorna dicionário com estatísticas da operação.
    """
    resultado = {
        "arquivo": caminho.name,
        "linhas_ok": 0,
        "linhas_corrigidas": 0,
        "linhas_erro": 0,
        "alterado": False,
    }

    texto = caminho.read_text(encoding="utf-8")
    linhas_raw = texto.strip().split("\n")

    if not linhas_raw:
        return resultado

    # Verificar header
    header_real = linhas_raw[0].strip().split(",")
    if header_real != HEADER_ESPERADO:
        resultado["linhas_erro"] = -1  # sinaliza schema incompatível
        return resultado

    saida = io.StringIO()
    writer = csv.writer(saida, quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
    writer.writerow(HEADER_ESPERADO)

    for linha in linhas_raw[1:]:
        if not linha.strip():
            continue
        parts = linha.split(",")
        reconstruida = reconstruir_linha(parts)

        if reconstruida is None:
            resultado["linhas_erro"] += 1
            writer.writerow(parts)  # mantém como estava
        elif len(parts) != N_COLUNAS:
            resultado["linhas_corrigidas"] += 1
            writer.writerow(reconstruida)
        else:
            resultado["linhas_ok"] += 1
            writer.writerow(parts)

    novo_conteudo = saida.getvalue()

    if novo_conteudo != texto.replace("\r\n", "\n"):
        resultado["alterado"] = True
        if CRIAR_BACKUP:
            shutil.copy2(caminho, pasta_backup / caminho.name)
        caminho.write_text(novo_conteudo, encoding="utf-8")

    return resultado


def main():
    pasta = Path(PASTA_CSV)
    if not pasta.exists():
        print(f"ERRO: pasta não encontrada → {pasta}")
        return

    csvs = sorted(pasta.glob("*.csv"))
    if not csvs:
        print("Nenhum arquivo .csv encontrado.")
        return

    pasta_backup = pasta / "_backup_pre_saneamento"
    if CRIAR_BACKUP:
        pasta_backup.mkdir(exist_ok=True)

    print(f"{'Arquivo':<45} {'OK':>5} {'Corrig':>7} {'Erro':>6} {'Alterado':>9}")
    print("-" * 75)

    total_corrigidas = 0
    total_erros = 0
    arquivos_alterados = 0

    for csv_path in csvs:
        r = sanear_arquivo(csv_path, pasta_backup)

        if r["linhas_erro"] == -1:
            print(f"{r['arquivo']:<45} {'--- schema incompatível ---':>30}")
            continue

        flag = "✓" if r["alterado"] else ""
        print(
            f"{r['arquivo']:<45} {r['linhas_ok']:>5} "
            f"{r['linhas_corrigidas']:>7} {r['linhas_erro']:>6} {flag:>9}"
        )

        total_corrigidas  += r["linhas_corrigidas"]
        total_erros       += r["linhas_erro"]
        arquivos_alterados += int(r["alterado"])

    print("-" * 75)
    print(f"Arquivos processados : {len(csvs)}")
    print(f"Arquivos alterados   : {arquivos_alterados}")
    print(f"Linhas corrigidas    : {total_corrigidas}")
    print(f"Linhas com erro      : {total_erros}")
    if CRIAR_BACKUP and arquivos_alterados:
        print(f"Backups em           : {pasta_backup}")


if __name__ == "__main__":
    main()


Arquivo                                          OK  Corrig   Erro  Alterado
---------------------------------------------------------------------------
2025_12_27_tribuna_liberal.csv                    2       2      0         ✓
2025_12_28_tribuna_liberal.csv                    2       2      0         ✓
2025_12_29_tribuna_liberal.csv                    3       2      0         ✓
2025_12_30_tribuna_liberal.csv                    2       0      0          
2025_12_31_tribuna_liberal.csv                    7       0      0          
2026_01_03_tribuna_liberal.csv                    6       2      0         ✓
2026_01_04_tribuna_liberal.csv                    4       2      0         ✓
2026_01_06_tribuna_liberal.csv                    4       1      0         ✓
2026_01_07_tribuna_liberal.csv                    6       0      0          
2026_01_08_tribuna_liberal.csv                    4       0      0          
2026_01_09_tribuna_liberal.csv                    4       1      0         ✓
